In [1]:
# ======================================================
# ✅ Setup común (Iris + escalado)
# ======================================================

import numpy as np
import seaborn as sns

from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

iris = sns.load_dataset("iris")
X = iris.drop(columns=["species"])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

def dbscan_summary_and_silhouette(labels, X):
    """Imprime resumen y retorna silhouette (o np.nan si no aplica)."""
    unique = np.unique(labels)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = np.sum(labels == -1)

    sil = np.nan
    if n_clusters >= 2:
        sil = silhouette_score(X, labels)

    print(f"clusters={n_clusters}, ruido={n_noise}, labels_unicos={unique}")
    print(f"silhouette_score={sil}")
    return sil

# 1) Parámetro: min_samples

**Qué es:** mínimo de puntos (incluyendo el punto) dentro de eps para que sea **core point**.  
**Tipo:** int  
**Default:** `min_samples=5`  

**Valores posibles:** int ≥ 1 (típicos: 3, 5, 10, 20; depende del tamaño del dataset).  
**Efecto:**
- min_samples ↑ → criterio más estricto → más ruido
- min_samples ↓ → criterio más permisivo → clusters más fáciles de formar

In [2]:
# ======================================================
# 🔹 Prueba min_samples (termina con silhouette_score)
# ======================================================

eps = 0.5
for ms in [3, 5, 8, 10, 15]:
    print(f"\n--- DBSCAN min_samples={ms} (eps={eps}, metric='euclidean') ---")
    model = DBSCAN(eps=eps, min_samples=ms, metric="euclidean")
    labels = model.fit_predict(X_scaled)
    silhouette_value = dbscan_summary_and_silhouette(labels, X_scaled)


--- DBSCAN min_samples=3 (eps=0.5, metric='euclidean') ---
clusters=7, ruido=17, labels_unicos=[-1  0  1  2  3  4  5  6]
silhouette_score=0.15971036873870176

--- DBSCAN min_samples=5 (eps=0.5, metric='euclidean') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN min_samples=8 (eps=0.5, metric='euclidean') ---
clusters=3, ruido=59, labels_unicos=[-1  0  1  2]
silhouette_score=0.18821743063720223

--- DBSCAN min_samples=10 (eps=0.5, metric='euclidean') ---
clusters=3, ruido=89, labels_unicos=[-1  0  1  2]
silhouette_score=0.009426257445526207

--- DBSCAN min_samples=15 (eps=0.5, metric='euclidean') ---
clusters=1, ruido=127, labels_unicos=[-1  0]
silhouette_score=nan


# 3) Parámetro: metric

**Qué es:** métrica de distancia para calcular vecinos.  
**Tipo:** str o callable  
**Default:** `metric='euclidean'`  

**Valores comunes (str):**
- `'euclidean'` (default)
- `'manhattan'`
- `'minkowski'` (requiere `p`)
- `'cosine'` (útil en embeddings/texto)

⚠️ Importante: la elección de métrica puede cambiar totalmente el resultado.


In [3]:
# ======================================================
# 🔹 Prueba metric (termina con silhouette_score)
# ======================================================

eps = 0.5
min_samples = 5

for met in ["euclidean", "manhattan", "cosine"]:
    print(f"\n--- DBSCAN metric='{met}' (eps={eps}, min_samples={min_samples}) ---")
    model = DBSCAN(eps=eps, min_samples=min_samples, metric=met)
    labels = model.fit_predict(X_scaled)
    silhouette_value = dbscan_summary_and_silhouette(labels, X_scaled)


--- DBSCAN metric='euclidean' (eps=0.5, min_samples=5) ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN metric='manhattan' (eps=0.5, min_samples=5) ---
clusters=4, ruido=110, labels_unicos=[-1  0  1  2  3]
silhouette_score=-0.21448771203070321

--- DBSCAN metric='cosine' (eps=0.5, min_samples=5) ---
clusters=1, ruido=0, labels_unicos=[0]
silhouette_score=nan


# 4) Parámetro: p (solo si metric='minkowski')

**Qué es:** exponente de Minkowski (define el tipo de distancia).  
**Tipo:** float  
**Default:** `p=2`  

**Valores típicos:**
- `p=1` → Manhattan (L1)
- `p=2` → Euclidean (L2)
- `p>2` → penaliza más diferencias grandes

In [4]:
# ======================================================
# 🔹 Prueba p con metric='minkowski' (termina con silhouette_score)
# ======================================================

eps = 0.5
min_samples = 5

for p in [1, 2, 3]:
    print(f"\n--- DBSCAN metric='minkowski', p={p} (eps={eps}, min_samples={min_samples}) ---")
    model = DBSCAN(eps=eps, min_samples=min_samples, metric="minkowski", p=p)
    labels = model.fit_predict(X_scaled)
    silhouette_value = dbscan_summary_and_silhouette(labels, X_scaled)  # <-- termina aquí


--- DBSCAN metric='minkowski', p=1 (eps=0.5, min_samples=5) ---
clusters=4, ruido=110, labels_unicos=[-1  0  1  2  3]
silhouette_score=-0.21448771203070321

--- DBSCAN metric='minkowski', p=2 (eps=0.5, min_samples=5) ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN metric='minkowski', p=3 (eps=0.5, min_samples=5) ---
clusters=2, ruido=27, labels_unicos=[-1  0  1]
silhouette_score=0.39892397031393567


# 5) Parámetro: algorithm

**Qué es:** algoritmo para búsqueda de vecinos.  
**Tipo:** str  
**Default:** `algorithm='auto'`  

**Valores posibles:**
- `'auto'`
- `'ball_tree'`
- `'kd_tree'`
- `'brute'`

Nota: el impacto principal suele ser rendimiento, no tanto el resultado (depende de la métrica).

In [5]:
# ======================================================
# 🔹 Prueba algorithm (termina con silhouette_score)
# ======================================================

eps = 0.5
min_samples = 5

for alg in ["auto", "ball_tree", "kd_tree", "brute"]:
    print(f"\n--- DBSCAN algorithm='{alg}' (eps={eps}, min_samples={min_samples}, metric='euclidean') ---")
    model = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean", algorithm=alg)
    labels = model.fit_predict(X_scaled)
    silhouette_value = dbscan_summary_and_silhouette(labels, X_scaled)  # <-- termina aquí


--- DBSCAN algorithm='auto' (eps=0.5, min_samples=5, metric='euclidean') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN algorithm='ball_tree' (eps=0.5, min_samples=5, metric='euclidean') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN algorithm='kd_tree' (eps=0.5, min_samples=5, metric='euclidean') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN algorithm='brute' (eps=0.5, min_samples=5, metric='euclidean') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726


# 6) Parámetro: leaf_size

**Qué es:** tamaño de hoja para BallTree/KDTree (impacta rendimiento).  
**Tipo:** int  
**Default:** `leaf_size=30`  

**Valores posibles:** int > 0 (típicos: 10, 30, 50, 100).

In [6]:
# ======================================================
# 🔹 Prueba leaf_size (termina con silhouette_score)
# ======================================================

eps = 0.5
min_samples = 5

for leaf in [10, 30, 50, 100]:
    print(f"\n--- DBSCAN leaf_size={leaf} (eps={eps}, min_samples={min_samples}, algorithm='ball_tree') ---")
    model = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean", algorithm="ball_tree", leaf_size=leaf)
    labels = model.fit_predict(X_scaled)
    silhouette_value = dbscan_summary_and_silhouette(labels, X_scaled)  # <-- termina aquí


--- DBSCAN leaf_size=10 (eps=0.5, min_samples=5, algorithm='ball_tree') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN leaf_size=30 (eps=0.5, min_samples=5, algorithm='ball_tree') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN leaf_size=50 (eps=0.5, min_samples=5, algorithm='ball_tree') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN leaf_size=100 (eps=0.5, min_samples=5, algorithm='ball_tree') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726


# 7) Parámetro: n_jobs

**Qué es:** número de hilos (paralelismo) para búsqueda de vecinos (cuando aplica).  
**Tipo:** int o None  
**Default:** `n_jobs=None`  

**Valores posibles:**
- `None` → default del entorno
- `-1` → usar todos los cores disponibles
- `1, 2, 4, ...` → número específico de cores

In [7]:
# ======================================================
# 🔹 Prueba n_jobs (termina con silhouette_score)
# ======================================================

eps = 0.5
min_samples = 5

for nj in [None, 1, -1]:
    print(f"\n--- DBSCAN n_jobs={nj} (eps={eps}, min_samples={min_samples}, metric='euclidean') ---")
    model = DBSCAN(eps=eps, min_samples=min_samples, metric="euclidean", n_jobs=nj)
    labels = model.fit_predict(X_scaled)
    silhouette_value = dbscan_summary_and_silhouette(labels, X_scaled)


--- DBSCAN n_jobs=None (eps=0.5, min_samples=5, metric='euclidean') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN n_jobs=1 (eps=0.5, min_samples=5, metric='euclidean') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726

--- DBSCAN n_jobs=-1 (eps=0.5, min_samples=5, metric='euclidean') ---
clusters=2, ruido=34, labels_unicos=[-1  0  1]
silhouette_score=0.35651648142700726
